In [1]:
# Import required libraries
import boto3
from pyathena import connect
import pandas as pd
import sagemaker

# Set up SageMaker session
sess = sagemaker.Session()
bucket = sess.default_bucket()
role = sagemaker.get_execution_role()
region = boto3.Session().region_name

/opt/conda/lib/python3.11/site-packages/pydantic/_internal/_fields.py:192: UserWarning: Field name "json" in "MonitoringDatasetFormat" shadows an attribute in parent "Base"
  warnings.warn(


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [2]:
# Define the database name and S3 staging directory for Athena query results
database_name = "dsoaws"  # or whatever you created in Notebook 02
s3_staging_dir = f"s3://{bucket}/athena/staging"

# Connect to Athena
conn = connect(region_name=region, s3_staging_dir=s3_staging_dir)

In [3]:
# S3 paths to the JSON files from gridAI.ipynb
s3_path_subregion = "s3://ecogridaidata/eia_electricity/eia_data_subregion_20250323_191047.json"
s3_path_energysource = "s3://ecogridaidata/eia_electricity/eia_data_energy_source_20250323_191047.json"

In [4]:
# Athena SQL to create subregion table
statement_subregion = f"""
CREATE EXTERNAL TABLE IF NOT EXISTS {database_name}.eia_subregion_data (
    period string,
    subba string,
    subba_name string,
    parent string,
    parent_name string,
    timezone string,
    value double,
    value_units string
)
ROW FORMAT SERDE 'org.openx.data.jsonserde.JsonSerDe'
LOCATION '{s3_path_subregion}'
TBLPROPERTIES ('classification'='json');
"""

print(statement_subregion)
pd.read_sql(statement_subregion, conn)


CREATE EXTERNAL TABLE IF NOT EXISTS dsoaws.eia_subregion_data (
    period string,
    subba string,
    subba_name string,
    parent string,
    parent_name string,
    timezone string,
    value double,
    value_units string
)
ROW FORMAT SERDE 'org.openx.data.jsonserde.JsonSerDe'
LOCATION 's3://ecogridaidata/eia_electricity/eia_data_subregion_20250323_191047.json'
TBLPROPERTIES ('classification'='json');



/tmp/ipykernel_3968/1078536749.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(statement_subregion, conn)


""


In [5]:
# Athena SQL to create energy source table
statement_energy = f"""
CREATE EXTERNAL TABLE IF NOT EXISTS {database_name}.eia_energy_source_data (
    period string,
    respondent string,
    respondent_name string,
    fueltype string,
    type_name string,
    timezone string,
    timezone_description string,
    value double,
    value_units string
)
ROW FORMAT SERDE 'org.openx.data.jsonserde.JsonSerDe'
LOCATION '{s3_path_energysource}'
TBLPROPERTIES ('classification'='json');
"""

print(statement_energy)
pd.read_sql(statement_energy, conn)


CREATE EXTERNAL TABLE IF NOT EXISTS dsoaws.eia_energy_source_data (
    period string,
    respondent string,
    respondent_name string,
    fueltype string,
    type_name string,
    timezone string,
    timezone_description string,
    value double,
    value_units string
)
ROW FORMAT SERDE 'org.openx.data.jsonserde.JsonSerDe'
LOCATION 's3://ecogridaidata/eia_electricity/eia_data_energy_source_20250323_191047.json'
TBLPROPERTIES ('classification'='json');



/tmp/ipykernel_3968/1460250323.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(statement_energy, conn)


""


In [6]:
# Check if tables are in the database
statement = f"SHOW TABLES IN {database_name}"
df_tables = pd.read_sql(statement, conn)
df_tables.head()

/tmp/ipykernel_3968/2528114882.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_tables = pd.read_sql(statement, conn)


,tab_name
0,amazon_reviews_tsv
1,eia_energy_source_data
2,eia_subregion_data


In [7]:
# Example query on the subregion table
query = f"""
SELECT * FROM {database_name}.eia_subregion_data
LIMIT 5
"""
df = pd.read_sql(query, conn)
df.head()

/tmp/ipykernel_3968/329312170.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ /opt/conda/lib/python3.11/site-packages/pandas/io/sql.py:2674 in execute                         │
│                                                                                                  │
│   2671 │   │   args = [] if params is None else [params]                                         │
│   2672 │   │   cur = self.con.cursor()                                                           │
│   2673 │   │   try:                                                                              │
│ ❱ 2674 │   │   │   cur.execute(sql, *args)                                                       │
│   2675 │   │   │   return cur                                                                    │
│   2676 │   │   except Exception as exc:                                                          │
│   2677 │   │   │   try:                                                                          │
│                                                                                                  │
│ /opt/conda/lib/python3.11/site-packages/pyathena/cursor.py:110 in execute                        │
│                                                                                                  │
│   107 │   │   │   │   self._retry_config,                                                        │
│   108 │   │   │   )                                                                              │
│   109 │   │   else:                                                                              │
│ ❱ 110 │   │   │   raise OperationalError(query_execution.state_change_reason)                    │
│   111 │   │   return self                                                                        │
│   112 │                                                                                          │
│   113 │   def executemany(                                                                       │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
OperationalError: com.amazonaws.services.s3.model.AmazonS3Exception: Access Denied (Service: Amazon S3; Status 
Code: 403; Error Code: AccessDenied; Request ID: DDT7NR6EG1NEBGH3; S3 Extended Request ID: 
k/ncQnRllbatNX4L8RK/cMiZck5lQoabwkhviqOExgMLBPXXCMYN261QDOcarc9njK1vzBYybCiLRjDF20X0gCZDrQ1If4UQ; Proxy: null), S3 
Extended Request ID: 
k/ncQnRllbatNX4L8RK/cMiZck5lQoabwkhviqOExgMLBPXXCMYN261QDOcarc9njK1vzBYybCiLRjDF20X0gCZDrQ1If4UQ (Bucket: 
ecogridaidata, Key: eia_electricity/eia_data_subregion_20250323_191047.json/)

During handling of the above exception, another exception occurred:

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ /opt/conda/lib/python3.11/site-packages/pandas/io/sql.py:2678 in execute                         │
│                                                                                                  │
│   2675 │   │   │   return cur                                                                    │
│   2676 │   │   except Exception as exc:                                                          │
│   2677 │   │   │   try:                                                                          │
│ ❱ 2678 │   │   │   │   self.con.rollback()                                                       │
│   2679 │   │   │   except Exception as inner_exc:  # pragma: no cover                            │
│   2680 │   │   │   │   ex = DatabaseError(                                                       │
│   2681 │   │   │   │   │   f"Execution failed on sql: {sql}\n{exc}\nunable to rollback"          │
│                                                                                                  │
│ /opt/conda/lib/python3.11/site-packages/pyathena/connection.py:368 in rollback                   │
│                                                                     